# Đo chi phí tính toán: APT + KAC và nhánh Spherical (4 dataset)

Notebook này **không train đầy đủ**. Với mỗi dataset, notebook train APT + KAC trên Task 1 trong vài epoch để đo tốc độ, rồi ngoại suy ra thời gian train cả chuỗi task. Sau đó đo chi phí của nhánh Spherical và chi phí suy luận của từng cấu hình.

| Phần | Đo gì |
|---|---|
| (a) Train APT + KAC | thời gian/epoch thực tế (gồm đọc ảnh), tốc độ GPU thuần (ảnh/giây), VRAM đỉnh, số tham số trainable (prompt + head KAC), ước tính giờ train cả chuỗi |
| (b) Xây tâm Spherical | thời gian trích đặc trưng + spherical k-means cho Task 1, ước tính cho cả chuỗi, bộ nhớ tâm khi đủ mọi lớp |
| (c) Suy luận | GFLOPs/ảnh, ms/ảnh, ảnh/giây, VRAM đỉnh cho: APT + KAC; + Spherical (APT stale / ViT đóng băng / Swin-B / Swin-T) |

Trọng số pretrained không ảnh hưởng thời gian, bộ nhớ hay FLOPs, nên mặc định `PRETRAINED = False` (không cần Internet nếu Kaggle đã có `timm`).
Kiến trúc giống repo APT (24 prompt cộng vào token CLS) và KAC (4 RBF); đây là bản tự cài để đo, không phải repo chính thức.

**Cách chạy:** bật GPU T4 → Add Input 4 dataset của nhóm → Run All (khoảng 15–25 phút).

## 1. Cấu hình

In [1]:
# ==============================================================================
# 1. CẤU HÌNH: đo chi phí tính toán của APT + KAC và nhánh Spherical (4 dataset)
#    Không train đầy đủ: mỗi dataset chỉ train Task 1 trong vài epoch để đo tốc độ, rồi ngoại suy.
# ==============================================================================
DATASETS = ["CIFAR100", "CUB200", "StanfordCars", "ImageNet_R"]
PHASE = "val"
SEED = 0                      # dùng thứ tự lớp + split Val của seed 0 (quy trình của nhóm)
PROFILE_EPOCHS = 2            # số epoch train Task 1 để đo (epoch đầu có warm-up, lấy số đo ở epoch cuối)
BATCH_TRAIN = 64              # = repo APT chính thức
BATCH_INFER = 64
WORKERS = 4                   # Kaggle có 4 CPU
K_CENTRES = 5                 # số tâm mỗi lớp (giá trị lớn nhất trong lưới -> chi phí lớn nhất)
N_WARM, N_REP = 3, 10         # số lần chạy nóng máy / số lần đo cho phần đo GPU thuần
# Siêu tham số APT theo experiments/*.sh của repo (chỉ dùng để ngoại suy thời gian train đầy đủ)
FULL_EPOCHS = {"CIFAR100": 30, "CUB200": 25, "StanfordCars": 50, "ImageNet_R": 30}
PROMPT_LR   = {"CIFAR100": 0.004, "CUB200": 0.02, "StanfordCars": 0.005, "ImageNet_R": 0.003}
HEAD_LR = 1e-3
VIT_NAME = "vit_base_patch16_224"
SWIN = {"swin_base_patch4_window7_224.ms_in22k_ft_in1k": "Swin-B",
        "swin_tiny_patch4_window7_224.ms_in1k": "Swin-T"}
# Trọng số không ảnh hưởng tới thời gian / bộ nhớ / FLOPs -> mặc định KHÔNG tải (không cần Internet).
PRETRAINED = False
INPUT_ROOT = "/kaggle/input"
OUT = "/kaggle/working/cost_profile"
SMOKE = False                 # True: chạy thử cực nhanh (chỉ để kiểm tra code)
import os; os.makedirs(OUT, exist_ok=True)
if SMOKE:
    PROFILE_EPOCHS, BATCH_TRAIN, BATCH_INFER, N_WARM, N_REP, WORKERS = 1, 4, 4, 1, 1, 0


## 2. Dữ liệu (cùng cách chia seed với các notebook của nhóm)

In [2]:
# ==============================================================================
# 2. DỮ LIỆU + CHIA SEED (y hệt check-dataset.ipynb / notebook APT của nhóm)
# ==============================================================================
import os, glob, json, pickle
import numpy as np
from torchvision.datasets import ImageFolder

DS_SPEC = {
    "CIFAR100":     dict(C=100, sizes=[10] * 10, kind="cifar",
                         roots=[f"{INPUT_ROOT}/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
                                f"{INPUT_ROOT}/cifar-100-python/cifar-100-python"]),
    "CUB200":       dict(C=200, sizes=[20] * 10, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/cyizhuo/cub-200-2011-by-classes-folder",
                                f"{INPUT_ROOT}/cub-200-2011-by-classes-folder"]),
    "StanfordCars": dict(C=196, sizes=[16] + [20] * 9, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
                                f"{INPUT_ROOT}/stanford-car-dataset-by-classes-folder/car_data/car_data"]),
    "ImageNet_R":   dict(C=200, sizes=[20] * 10, kind="folder",
                         roots=[f"{INPUT_ROOT}/datasets/thohintrnhngc/imagenet-r", f"{INPUT_ROOT}/imagenet-r"]),
}

def find_root(ds):
    sp = DS_SPEC[ds]
    ok = (lambda p: all(os.path.isfile(os.path.join(p, f)) for f in ("train", "test", "meta"))) if sp["kind"] == "cifar" \
         else (lambda p: os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test")))
    for p in sp["roots"]:
        if ok(p): return p
    tail = os.path.basename(sp["roots"][0].rstrip("/"))
    for p in glob.glob(f"{INPUT_ROOT}/**/{tail}", recursive=True):
        if ok(p): return p
    raise FileNotFoundError(f"{ds}: không thấy dữ liệu. Add Input dataset của nhóm ({sp['roots'][0]})")

def setup_seed(train_targets, seed, num_classes, task_sizes, n_splits=5, base_seed=1993):
    class_order = np.random.RandomState(seed).permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur: cur + sz]); cur += sz
    targets = np.asarray(train_targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        perm = rng_split.permutation(np.where(targets == c)[0])
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed: tr_idx.extend(folds[k])
    return task_classes, np.array(tr_idx), np.array(va_idx)

SRC = {}
def load_source(ds):
    """S["tr"], S["te"]: đường dẫn ảnh (folder) hoặc mảng ảnh uint8 (CIFAR); S["tr_id"]: tên ảnh dùng để đối chiếu json."""
    if ds in SRC: return SRC[ds]
    root, sp = find_root(ds), DS_SPEC[ds]
    if sp["kind"] == "cifar":
        S = dict(root=root)
        for s, name in (("tr", "train"), ("te", "test")):
            with open(os.path.join(root, name), "rb") as f: e = pickle.load(f, encoding="latin1")
            S[s] = e["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1); S[s + "_y"] = [int(v) for v in e["fine_labels"]]
            S[s + "_id"] = [f"{name}:{i}" for i in range(len(S[s + "_y"]))]
    else:
        tr, te = ImageFolder(os.path.join(root, "train")), ImageFolder(os.path.join(root, "test"))
        assert tr.classes == te.classes, f"{ds}: tên lớp train/test không khớp"
        S = dict(root=root, tr=[p for p, _ in tr.samples], tr_y=list(tr.targets), te=[p for p, _ in te.samples], te_y=list(te.targets))
        for s in ("tr", "te"): S[s + "_id"] = [os.path.relpath(p, root) for p in S[s]]
    assert len(set(S["tr_y"])) == sp["C"], f"{ds}: số lớp {len(set(S['tr_y']))} ≠ {sp['C']}"
    SRC[ds] = S
    print(f"{ds}: {root} | train {len(S['tr_y'])} ảnh, test {len(S['te_y'])} ảnh")
    return S

def seed_split(ds, seed):
    """(train_items, train_labels, eval_items, eval_labels) theo đúng thứ tự file json của nhóm.
    item = (split, chỉ số ảnh), split ∈ {"tr","te"}; nhãn = vị trí lớp trong thứ tự task."""
    sp, S = DS_SPEC[ds], load_source(ds)
    task_classes, tr_idx, va_idx = setup_seed(S["tr_y"], seed, sp["C"], sp["sizes"])
    pos = {c: i for i, c in enumerate(sum(task_classes, []))}
    if PHASE == "val":
        tr = [("tr", int(i)) for i in tr_idx]; ev = [("tr", int(i)) for i in va_idx]
    else:
        tr = [("tr", i) for i in range(len(S["tr_y"]))]; ev = [("te", i) for i in range(len(S["te_y"]))]
    lab = lambda it: pos[S[it[0] + "_y"][it[1]]]
    return tr, [lab(x) for x in tr], ev, [lab(x) for x in ev]

def check_against_json(ds, seed, ev, ev_y):
    """Nếu output phiên APT (thư mục splits) được Add Input: so danh sách ảnh đánh giá với file json đã dùng khi train KAC."""
    hits = glob.glob(f"{INPUT_ROOT}/**/splits/{ds}/{PHASE}/seed{seed}/test.json", recursive=True)
    if not hits: return "không có json để đối chiếu"
    js = json.load(open(hits[0])); S = load_source(ds)
    rel = (lambda p: p) if DS_SPEC[ds]["kind"] == "cifar" else \
          (lambda p: os.path.relpath(p, js["root"]) if p.startswith(js["root"]) else p)
    mine = [S[s + "_id"][i] for s, i in ev]
    assert [rel(p) for p in js["data"]] == mine and js["targets"] == ev_y, f"{ds} seed {seed}: split khác file json của phiên APT"
    return "khớp file json của phiên APT"


## 3. Mô hình và hàm đo

In [3]:
# ==============================================================================
# 3. MÔ HÌNH: ViT-B/16 đóng băng + prompt cộng vào CLS (APT) + head KAC, và nhánh Spherical
#    Kiến trúc giống repo APT (24 prompt = 12 tầng x {K, V}, cộng vào token CLS) và KAC (4 RBF).
# ==============================================================================
import time, math, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode
import torchvision.transforms as T

DEV = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU"
print("thiết bị:", GPU_NAME)
if DEV == "cuda":
    torch.backends.cudnn.benchmark = True

class Attention(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__(); self.heads, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, dim * 3); self.proj = nn.Linear(dim, dim)
    def forward(self, x, prompt=None):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, self.hd).permute(2, 0, 3, 1, 4).unbind(0)
        if prompt is not None:
            pk, pv = prompt
            k = torch.cat([k[:, :, :1] + pk.to(k.dtype), k[:, :, 1:]], dim=2)
            v = torch.cat([v[:, :, :1] + pv.to(v.dtype), v[:, :, 1:]], dim=2)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))

class Block(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim, eps=1e-6), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x, prompt=None):
        x = x + self.attn(self.norm1(x), prompt)
        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):
    def __init__(self, dim=768, depth=12, heads=12):
        super().__init__(); self.depth, self.heads = depth, heads
        self.patch_embed = nn.Conv2d(3, dim, 16, 16)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim)); self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)]); self.norm = nn.LayerNorm(dim, eps=1e-6)
    def forward(self, x, prompt_fn=None):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], 1) + self.pos_embed
        for i, b in enumerate(self.blocks): x = b(x, prompt_fn(i) if prompt_fn else None)
        return self.norm(x)[:, 0]

def make_vit():
    vit = ViT()
    if PRETRAINED:                                   # chỉ cần khi muốn trọng số thật (không ảnh hưởng chi phí)
        ref = timm.create_model(VIT_NAME, pretrained=True, num_classes=0)
        sd = {k.replace("mlp.fc1", "mlp.0").replace("mlp.fc2", "mlp.2").replace("patch_embed.proj", "patch_embed"): v
              for k, v in ref.state_dict().items() if not k.startswith("head")}
        print(vit.load_state_dict(sd, strict=False)); del ref
    for p in vit.parameters(): p.requires_grad_(False)
    return vit.eval()

class APTPrompt(nn.Module):
    def __init__(self, depth=12, dim=768, heads=12, ema=0.7):
        super().__init__(); self.heads, self.hd, self.ema = heads, dim // heads, ema
        self.prompt_tokens = nn.Parameter(torch.randn(depth * 2, dim) * 0.02)
        self.register_buffer("merged", torch.zeros(depth * 2, dim)); self.n_merged = 0; self.use_merged = False
    def get(self, l):
        P = self.merged if self.use_merged else self.prompt_tokens
        return P[2 * l].view(self.heads, 1, self.hd), P[2 * l + 1].view(self.heads, 1, self.hd)
    @torch.no_grad()
    def merge(self):
        self.merged.copy_(self.prompt_tokens if self.n_merged == 0 else self.merged * self.ema + self.prompt_tokens * (1 - self.ema))
        self.n_merged += 1

class KACHead(nn.Module):
    def __init__(self, dim, n_cls, num_grids=4, grid_range=(-2.0, 2.0)):
        super().__init__(); gmin, gmax = grid_range
        self.layernorm = nn.LayerNorm(dim); self.register_buffer("grid", torch.linspace(gmin, gmax, num_grids))
        self.den = (gmax - gmin) / (num_grids - 1)
        self.spline_linear = nn.Linear(dim * num_grids, n_cls, bias=False); nn.init.zeros_(self.spline_linear.weight)
    def forward(self, f):
        x = self.layernorm(f)
        return self.spline_linear(torch.exp(-((x[..., None] - self.grid) / self.den) ** 2).flatten(1))

class APTKAC(nn.Module):
    def __init__(self, vit, n_cls):
        super().__init__(); self.vit, self.prompt, self.head = vit, APTPrompt(), KACHead(768, n_cls)
    def features(self, x):
        self.prompt.use_merged = (not self.training) and self.prompt.n_merged > 0
        with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == "cuda")):
            return self.vit(x, prompt_fn=self.prompt.get).float()
    def forward(self, x, return_feat=False):
        f = self.features(x); logit = self.head(f)
        return (logit, f) if return_feat else logit

# ---------------- nhánh Spherical (giống fusion_online.py của nhóm) ----------------
def embed(f): return F.normalize(F.layer_norm(f.float(), (f.shape[1],)), dim=1)

def sph_kmeans(Z, k, g, iters=20):
    k = min(k, Z.shape[0]); C = Z[torch.randperm(Z.shape[0], generator=g)[:k]].clone()
    for _ in range(iters):
        a = (Z @ C.T).argmax(1); newC = torch.zeros_like(C).index_add_(0, a, Z)
        empty = newC.norm(dim=1) < 1e-8
        if empty.any(): newC[empty] = Z[torch.randint(0, Z.shape[0], (int(empty.sum()),), generator=g)]
        C = F.normalize(newC, dim=1)
    return C

def sph_fuse(logit, f, cen, kappa, w=0.5):
    """fuse: (1-w) log P_KAC + w log P_sph trên mọi lớp đã học; cen: (C, k, D)."""
    z = embed(f); cos = torch.einsum("nd,ckd->nck", z, cen)
    s = torch.logsumexp(kappa * (cos - 1), dim=2)
    return ((1 - w) * F.log_softmax(logit.float(), 1) + w * F.log_softmax(s, 1)).argmax(1)

class Imgs(Dataset):
    def __init__(self, src, ys, tf): self.src, self.ys, self.tf = src, ys, tf
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        x = self.src[i]
        if isinstance(x, str):
            with Image.open(x) as im: x = im.convert("RGB")
        else: x = Image.fromarray(x)
        return self.tf(x), self.ys[i]

TRAIN_TF = T.Compose([T.RandomResizedCrop(224), T.RandomHorizontalFlip(), T.ToTensor()])
TEST_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])

def sync():
    if DEV == "cuda": torch.cuda.synchronize()

def time_fn(fn):
    """giây / lần gọi (đo GPU thuần, dữ liệu đã nằm sẵn trên GPU)."""
    for _ in range(N_WARM): fn()
    sync(); t0 = time.time()
    for _ in range(N_REP): fn()
    sync(); return (time.time() - t0) / N_REP

def reset_peak():
    if DEV == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_gb(): return torch.cuda.max_memory_allocated() / 1024 ** 3 if DEV == "cuda" else float("nan")

from torch.nn.attention import sdpa_kernel, SDPBackend
def gflops(fn):
    # ép attention chạy đường MATH để FlopCounter đếm cả phép nhân QK^T và AV (kernel fused có thể không được đếm)
    with sdpa_kernel(SDPBackend.MATH), FlopCounterMode(display=False) as fc: fn()
    return fc.get_total_flops() / 2e9          # FLOPs/2 = MACs, cùng quy ước với bảng backbone (17.56 G cho ViT-B)


thiết bị: Tesla T4


## 4. Đo chi phí

In [4]:
# ==============================================================================
# 4. ĐO CHI PHÍ CHO TỪNG DATASET
#    (a) Train APT + KAC trên Task 1: thời gian/epoch thực tế (gồm đọc ảnh), tốc độ GPU thuần, VRAM đỉnh, tham số trainable
#    (b) Nhánh Spherical: thời gian trích đặc trưng + spherical k-means cho Task 1, bộ nhớ tâm của mọi lớp
#    (c) Suy luận: GFLOPs/ảnh, ms/ảnh, VRAM đỉnh cho từng cấu hình
# ==============================================================================
VIT = make_vit().to(DEV)
SWIN_M = {}
def get_swin(name):
    if name not in SWIN_M:
        SWIN_M[name] = timm.create_model(name, pretrained=PRETRAINED, num_classes=0).eval().to(DEV)
        for p in SWIN_M[name].parameters(): p.requires_grad_(False)
    return SWIN_M[name]

TRAIN_ROWS, SPH_ROWS, INF_ROWS = [], [], []
for ds in DATASETS:
    print(f"\n{'=' * 70}\n{ds}\n{'=' * 70}", flush=True)
    sp, S = DS_SPEC[ds], load_source(ds)
    task_classes, tr_idx, va_idx = setup_seed(S["tr_y"], SEED, sp["C"], sp["sizes"])
    pos = {c: i for i, c in enumerate(sum(task_classes, []))}
    c1 = set(task_classes[0]); lo, hi = 0, sp["sizes"][0]
    idx1 = [int(i) for i in tr_idx if S["tr_y"][i] in c1]
    if SMOKE: idx1 = idx1[:8]
    items, ys = [S["tr"][i] for i in idx1], [pos[S["tr_y"][i]] for i in idx1]
    n_train_all, n_val_all, T_ = len(tr_idx), len(va_idx), len(sp["sizes"])

    # ---------------- (a) train APT + KAC trên Task 1 ----------------
    model = APTKAC(VIT, sp["C"]).to(DEV)
    n_prompt = model.prompt.prompt_tokens.numel(); n_head = sum(p.numel() for p in model.head.parameters())
    opt = torch.optim.Adam([{"params": [model.prompt.prompt_tokens], "lr": PROMPT_LR[ds]},
                            {"params": list(model.head.parameters()), "lr": HEAD_LR}])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))
    def train_step(x, y):
        loss = F.cross_entropy(model(x)[:, lo:hi], y - lo)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    dl = DataLoader(Imgs(items, ys, TRAIN_TF), batch_size=BATCH_TRAIN, shuffle=True, drop_last=len(items) > BATCH_TRAIN,
                    num_workers=WORKERS, pin_memory=(DEV == "cuda"), persistent_workers=WORKERS > 0)
    model.train(); reset_peak(); ep_t, n_seen = [], 0
    for ep in range(PROFILE_EPOCHS):
        sync(); t0 = time.time(); n_seen = 0
        for x, y in dl:
            train_step(x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)); n_seen += len(y)
        sync(); ep_t.append(time.time() - t0)
        print(f"  train Task 1 epoch {ep + 1}/{PROFILE_EPOCHS}: {ep_t[-1]:.1f}s ({n_seen} ảnh)", flush=True)
    vram_train = peak_gb()
    real_ips = n_seen / ep_t[-1]
    xb = torch.randn(BATCH_TRAIN, 3, 224, 224, device=DEV); yb = torch.randint(lo, hi, (BATCH_TRAIN,), device=DEV)
    gpu_ips = BATCH_TRAIN / time_fn(lambda: train_step(xb, yb))
    model.prompt.merge(); model.eval()
    TRAIN_ROWS.append(dict(dataset=ds, GPU=GPU_NAME, task1_images=len(items), epoch_time_s=ep_t[-1],
        train_img_per_s_real=real_ips, train_img_per_s_gpu=gpu_ips, peak_VRAM_train_GB=vram_train,
        trainable_prompt=n_prompt, trainable_KAC_head=n_head, trainable_total=n_prompt + n_head,
        train_images_all_tasks=n_train_all, full_epochs=FULL_EPOCHS[ds],
        est_full_train_h_real=n_train_all * FULL_EPOCHS[ds] / real_ips / 3600,
        est_full_train_h_gpu=n_train_all * FULL_EPOCHS[ds] / gpu_ips / 3600))

    # ---------------- (b) nhánh Spherical: trích đặc trưng Task 1 + k-means ----------------
    dl_feat = DataLoader(Imgs(items, ys, TEST_TF), batch_size=BATCH_INFER, shuffle=False,
                         num_workers=WORKERS, pin_memory=(DEV == "cuda"))
    # APT (stale) cũng cần 1 lượt forward (có prompt) trên ảnh train của task vừa học để lấy tâm; lúc suy luận thì dùng lại đặc trưng sẵn có
    sources = [("ViT-B/16 đóng băng", lambda x: VIT(x)), ("APT (stale)", lambda x: model.features(x))] + \
              [(lab, (lambda x, n=n: get_swin(n)(x))) for n, lab in SWIN.items()]
    CEN = {}
    for lab, fwd in sources:
        sync(); t0 = time.time(); fs = []
        with torch.no_grad():
            for x, _ in dl_feat:
                with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == "cuda")):
                    fs.append(fwd(x.to(DEV, non_blocking=True)).float())
        sync(); t_feat = time.time() - t0
        Z, Y = embed(torch.cat(fs)), torch.tensor(ys, device=DEV); D = Z.shape[1]
        g = torch.Generator().manual_seed(0); sync(); t0 = time.time()
        for c in range(lo, hi):
            Zc = Z[Y == c]
            if len(Zc): sph_kmeans(Zc.cpu(), K_CENTRES, g)        # code nhóm chạy k-means trên CPU
        z0 = Z[:512]; kappa = 16.0 / (1 - z0 @ z0.T)[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool, device=DEV), 1)].median().item()
        sync(); t_km = time.time() - t0
        CEN[lab] = (F.normalize(torch.randn(sp["C"], K_CENTRES, D, device=DEV), dim=2), kappa)   # đủ C lớp như cuối chuỗi task
        SPH_ROWS.append(dict(dataset=ds, source=lab, feat_dim=D, task1_images=len(items),
            feat_time_task1_s=t_feat, kmeans_time_task1_s=t_km,
            est_build_all_tasks_s=t_feat * n_train_all / max(len(items), 1) + t_km * T_,
            centres_MB_all_classes=sp["C"] * K_CENTRES * D * 4 / 1024 ** 2))
        print(f"  Spherical {lab}: trích {t_feat:.1f}s, k-means {t_km:.2f}s cho Task 1", flush=True)

    # ---------------- (c) suy luận: GFLOPs, ms/ảnh, VRAM ----------------
    model.eval(); x1 = torch.randn(1, 3, 224, 224, device=DEV); xb = torch.randn(BATCH_INFER, 3, 224, 224, device=DEV)
    def cfg_apt(x): return model(x).argmax(1)
    def cfg_apt_stale(x):
        logit, f = model(x, return_feat=True); cen, kp = CEN["APT (stale)"]; return sph_fuse(logit, f, cen, kp)
    def make_frozen(lab, fwd):
        def run(x):
            logit = model(x); cen, kp = CEN[lab]
            with torch.autocast(DEV, dtype=torch.float16, enabled=(DEV == "cuda")): f = fwd(x).float()
            return sph_fuse(logit, f, cen, kp)
        return run
    configs = [("APT + KAC (baseline)", cfg_apt, 0.0), ("+ Spherical, APT (stale)", cfg_apt_stale, 0.0),
               ("+ Spherical, ViT-B/16 đóng băng", make_frozen("ViT-B/16 đóng băng", lambda x: VIT(x)), 0.0)] + \
              [(f"+ Spherical, {lab}", make_frozen(lab, (lambda x, n=n: get_swin(n)(x))),
                sum(p.numel() for p in get_swin(n).parameters()) / 1e6) for n, lab in SWIN.items()]
    with torch.no_grad():
        for name, fn, extra in configs:
            gf = gflops(lambda: fn(x1)); reset_peak()
            sec = time_fn(lambda: fn(xb)); vram = peak_gb()
            INF_ROWS.append(dict(dataset=ds, config=name, GFLOPs_per_img=gf, ms_per_img=1000 * sec / BATCH_INFER,
                                 img_per_s=BATCH_INFER / sec, peak_VRAM_infer_GB=vram, extra_backbone_params_M=extra,
                                 extra_trainable_params=0))
            print(f"  suy luận {name}: {gf:.2f} GFLOPs/ảnh, {1000 * sec / BATCH_INFER:.2f} ms/ảnh", flush=True)
    del model, opt; reset_peak()



CIFAR100
CIFAR100: /kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python | train 50000 ảnh, test 10000 ảnh
  train Task 1 epoch 1/2: 27.2s (3968 ảnh)
  train Task 1 epoch 2/2: 21.8s (3968 ảnh)
  Spherical ViT-B/16 đóng băng: trích 9.8s, k-means 0.31s cho Task 1
  Spherical APT (stale): trích 10.4s, k-means 0.07s cho Task 1
  Spherical Swin-B: trích 20.8s, k-means 0.09s cho Task 1
  Spherical Swin-T: trích 9.8s, k-means 0.07s cho Task 1
  suy luận APT + KAC (baseline): 17.56 GFLOPs/ảnh, 2.78 ms/ảnh
  suy luận + Spherical, APT (stale): 17.56 GFLOPs/ảnh, 2.79 ms/ảnh
  suy luận + Spherical, ViT-B/16 đóng băng: 35.13 GFLOPs/ảnh, 5.53 ms/ảnh
  suy luận + Spherical, Swin-B: 32.99 GFLOPs/ảnh, 7.80 ms/ảnh
  suy luận + Spherical, Swin-T: 22.05 GFLOPs/ảnh, 5.09 ms/ảnh

CUB200
CUB200: /kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder | train 5994 ảnh, test 5794 ảnh
  train Task 1 epoch 1/2: 12.3s (448 ảnh)
  train Task 1 epoch 2/2: 3.4s (448 ảnh)
  Spherical ViT-B/16 

## 5. Bảng kết quả

In [5]:
# ==============================================================================
# 5. BẢNG KẾT QUẢ (lưu CSV trong OUT)
# ==============================================================================
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
tr, sph, inf = pd.DataFrame(TRAIN_ROWS), pd.DataFrame(SPH_ROWS), pd.DataFrame(INF_ROWS)
tr.to_csv(f"{OUT}/cost_train.csv", index=False, float_format="%.4f")
sph.to_csv(f"{OUT}/cost_spherical_build.csv", index=False, float_format="%.4f")
inf.to_csv(f"{OUT}/cost_inference.csv", index=False, float_format="%.4f")
print("GPU:", GPU_NAME, "| batch train", BATCH_TRAIN, "| batch suy luận", BATCH_INFER, "| fp16 | ảnh 224x224")
print("\n(a) Train APT + KAC (đo trên Task 1, ngoại suy cho toàn bộ chuỗi task):")
print(tr[["dataset", "task1_images", "epoch_time_s", "train_img_per_s_real", "train_img_per_s_gpu", "peak_VRAM_train_GB",
          "trainable_total", "est_full_train_h_real", "est_full_train_h_gpu"]].round(3).to_string(index=False))
print("\n(b) Xây tâm Spherical (Task 1; ước tính cho cả chuỗi; bộ nhớ tâm khi đủ mọi lớp):")
print(sph[["dataset", "source", "feat_time_task1_s", "kmeans_time_task1_s", "est_build_all_tasks_s", "centres_MB_all_classes"]].round(3).to_string(index=False))
print("\n(c) Suy luận (GPU thuần, dữ liệu đã trên GPU):")
print(inf[["dataset", "config", "GFLOPs_per_img", "ms_per_img", "img_per_s", "peak_VRAM_infer_GB", "extra_backbone_params_M"]].round(3).to_string(index=False))


GPU: Tesla T4 | batch train 64 | batch suy luận 64 | fp16 | ảnh 224x224

(a) Train APT + KAC (đo trên Task 1, ngoại suy cho toàn bộ chuỗi task):
     dataset  task1_images  epoch_time_s  train_img_per_s_real  train_img_per_s_gpu  peak_VRAM_train_GB  trainable_total  est_full_train_h_real  est_full_train_h_gpu
    CIFAR100          4000        21.789               182.108              175.362               3.730           327168                  1.830                 1.901
      CUB200           479         3.437               130.339              151.476               4.239           634368                  0.255                 0.220
StanfordCars           538         4.419               115.871              144.476               4.221           622080                  0.772                 0.619
  ImageNet_R          1926        13.603               141.144              143.731               4.222           634368                  1.129                 1.109

(b) Xây tâm Spherical (T

In [6]:
import zipfile, glob
with zipfile.ZipFile(f"{OUT}/cost_profile_results.zip", "w") as z:
    for p in glob.glob(f"{OUT}/*.csv"): z.write(p, os.path.basename(p))
print("Tải file này về:", f"{OUT}/cost_profile_results.zip")


Tải file này về: /kaggle/working/cost_profile/cost_profile_results.zip
